## Hybrid Search Langchain

In [6]:
! pip install --upgrade --quiet pinecone pinecone-text pinecone-notebooks


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import os 
from dotenv import load_dotenv
load_dotenv()
api_key=os.getenv('PINECONE_API_KEY')

In [13]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [14]:
import os 
from pinecone import Pinecone,ServerlessSpec 
index_name="hybrid-search-langchain-pinecone"
## initialize the Pinecone client 
pc=Pinecone(api_key=api_key)

#create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,#dimension of dense vector
        metric='dotproduct',# sparse values supported only for dotproduct
        spec=ServerlessSpec(cloud='aws',region='us-east-1')
    )

In [15]:
index=pc.Index(index_name)
index

/Users/shindesudeep/Desktop/GenAI_RLHF/Langchain/genai_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
## vector embedding and sparse matrix
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [18]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder=BM25Encoder().default()
bm25_encoder

In [19]:
sentences=[
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]
##Tfidf values on the sentences
bm25_encoder.fit(sentences)

## store the values to a json file 
bm25_encoder.dump("bm25_values.json")

#load to your BM25Encoder object 
bm25_encoder=BM25Encoder().load('bm25_values.json')


100%|██████████| 3/3 [00:00<00:00, 176.77it/s]


In [20]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)


In [21]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x31cfa7b60>, index=<pinecone.db_data.index.Index object at 0x1086b2fc0>)

In [23]:
retriever.add_texts([
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
])

100%|██████████| 1/1 [00:03<00:00,  3.49s/it]


In [24]:
retriever.invoke("What city did i visit last")

[Document(metadata={'score': 0.27071774}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.255977273}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.235791892}, page_content='In 2023, I visited Paris')]